# 02. QLoRA Fine-Tuning (Few-Shot)

Qwen3-1.7B 베이스 모델에 4-bit NF4 양자화 + LoRA (r=32) 를 적용한  
구조적 Few-Shot SFT 실험 결과입니다.  
스크립트: `src/04_qlora_finetuning_few_shot.py`

In [1]:
print('=== QLoRA Configuration ===')
config = {
    'Base model'      : 'Qwen3-1.7B',
    'Quantization'    : '4-bit NF4 (bitsandbytes)',
    'LoRA rank (r)'   : 32,
    'LoRA alpha'      : 16,
    'LoRA dropout'    : 0.05,
    'Target modules'  : 'q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj',
    'Max seq length'  : '1,408',
    'Effective batch' : '64  (per_device=2 × grad_accum=32)',
    'Optimizer'       : 'paged_adamw_8bit',
    'Learning rate'   : '2e-4  (cosine schedule, warmup_ratio=0.03)',
    'Epochs'          : 1,
    'Train samples'   : '119,918',
}
for k, v in config.items():
    print(f'{k:<18}: {v}')

=== QLoRA Configuration ===
Base model        : Qwen3-1.7B
Quantization      : 4-bit NF4 (bitsandbytes)
LoRA rank (r)     : 32
LoRA alpha        : 16
LoRA dropout      : 0.05
Target modules    : q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
Max seq length    : 1,408
Effective batch   : 64  (per_device=2 × grad_accum=32)
Optimizer         : paged_adamw_8bit
Learning rate     : 2e-4  (cosine schedule, warmup_ratio=0.03)
Epochs            : 1
Train samples     : 119,918


In [2]:
print('=== VRAM Usage (Peak) ===')
print('Standard LoRA (BF16, 16-bit) : 4.30 GB')
print('QLoRA        (NF4,  4-bit)   : 2.15 GB  ← 50.0% reduction')
print()
print('Training speed:')
print('  Standard LoRA : 9.83 samples/s')
print('  QLoRA         : 6.75 samples/s  (31% slower due to dequantization overhead)')

=== VRAM Usage (Peak) ===
Standard LoRA (BF16, 16-bit) : 4.30 GB
QLoRA        (NF4,  4-bit)   : 2.15 GB  ← 50.0% reduction

Training speed:
  Standard LoRA : 9.83 samples/s
  QLoRA         : 6.75 samples/s  (31% slower due to dequantization overhead)


In [3]:
import pandas as pd

log_data = [
    (50,    0.03, 1.4633, 1.719e-04, 0.7238),
    (100,   0.05, 0.1295, 1.997e-04, 0.9743),
    (200,   0.11, 0.1005, 1.970e-04, 0.9783),
    (400,   0.21, 0.0920, 1.830e-04, 0.9796),
    (600,   0.32, 0.0886, 1.592e-04, 0.9801),
    (800,   0.43, 0.0850, 1.200e-04, 0.9806),
    (1000,  0.53, 0.0846, 9.421e-05, 0.9807),
    (1200,  0.64, 0.0837, 6.072e-05, 0.9810),
    (1400,  0.75, 0.0829, 3.187e-05, 0.9811),
    (1600,  0.85, 0.0825, 1.109e-05, 0.9811),
    (1874,  0.99, 0.0822, 9.341e-08, 0.9812),
]

print('=== Training Log (selected steps) ===')
print(f' {"step":>4} | {"epoch":>5} | {"train_loss":>10} | {"lr":<12} | {"token_acc"}')
print('-'*6+'|'+'-'*7+'|'+'-'*12+'|'+'-'*14+'|'+'-'*10)
for step, epoch, loss, lr, acc in log_data:
    print(f' {step:>4} |  {epoch:.2f} |     {loss:.4f} | {lr:.3e}    |   {acc*100:.2f}%')

print(f'\nLoss dropped from 1.463 → 0.082 within 100 steps (rapid convergence).')

=== Training Log (selected steps) ===
 step | epoch | train_loss | lr           | token_acc
------|-------|------------|--------------|----------
   50 |  0.03 |     1.4633 | 1.719e-04    |   72.38%
  100 |  0.05 |     0.1295 | 1.997e-04    |   97.43%
  200 |  0.11 |     0.1005 | 1.970e-04    |   97.83%
  400 |  0.21 |     0.0920 | 1.830e-04    |   97.96%
  600 |  0.32 |     0.0886 | 1.592e-04    |   98.01%
  800 |  0.43 |     0.0850 | 1.200e-04    |   98.06%
 1000 |  0.53 |     0.0846 | 9.421e-05    |   98.07%
 1200 |  0.64 |     0.0837 | 6.072e-05    |   98.10%
 1400 |  0.75 |     0.0829 | 3.187e-05    |   98.11%
 1600 |  0.85 |     0.0825 | 1.109e-05    |   98.11%
 1874 |  0.99 |     0.0822 | 9.341e-08    |   98.12%

Loss dropped from 1.463 → 0.082 within 100 steps (rapid convergence).


In [4]:
eval_data = [
    (200,  0.11, 0.09885, 0.97859),
    (400,  0.21, 0.09202, 0.97968),
    (600,  0.32, 0.08905, 0.98013),
    (800,  0.43, 0.08692, 0.98048),
    (1000, 0.53, 0.08516, 0.98073),
    (1200, 0.64, 0.08404, 0.98089),
    (1400, 0.75, 0.08301, 0.98105),
    (1600, 0.85, 0.08249, 0.98113),
    (1874, 0.96, 0.08231, 0.98115),
]

print('=== Validation Loss (eval every 200 steps) ===')
print(f' {"step":>4} | {"epoch":>5} | {"eval_loss":>9} | eval_token_acc')
print('-'*6+'|'+'-'*7+'|'+'-'*11+'|'+'-'*15)
for i, (step, epoch, eloss, eacc) in enumerate(eval_data):
    marker = '  ← best checkpoint' if i == len(eval_data)-1 else ''
    print(f' {step:>4} |  {epoch:.2f} |    {eloss:.4f} |        {eacc*100:.2f}%{marker}')

=== Validation Loss (eval every 200 steps) ===
 step | epoch | eval_loss | eval_token_acc
------|-------|-----------|---------------
  200 |  0.11 |    0.0989 |        97.86%
  400 |  0.21 |    0.0920 |        97.97%
  600 |  0.32 |    0.0891 |        98.01%
  800 |  0.43 |    0.0869 |        98.05%
 1000 |  0.53 |    0.0852 |        98.07%
 1200 |  0.64 |    0.0840 |        98.09%
 1400 |  0.75 |    0.0830 |        98.10%
 1600 |  0.85 |    0.0825 |        98.11%
 1874 |  0.96 |    0.0823 |        98.11%  ← best checkpoint
